# Systematic Review
**References:**
* ❌ too old - https://github.com/chandraveshchaudhari/systematic-reviewpy
* ❌ agentic AI - https://github.com/PouriaRouzrokh/LatteReview
* ✅ Uses PubMed API - https://github.com/gijswobben/pymed

**TO-DO**
* ✅ Clean this notebook up
* Run the query on other databases as well
* QC the results from each database, starting with pubmed.
* Once QC'd, merge the results across each database result and process.

**REFERENCES THAT CAUGHT MY EYE**
* https://pubmed.ncbi.nlm.nih.gov/33202965/
* https://pubmed.ncbi.nlm.nih.gov/38389433/

## Environment Setup

In [1]:
# Import all required packages
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path
from pymed import PubMed
import json
import os
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Load environment variables
load_dotenv()

True

# Specify Search Strategy

Hack: I used the query builder on https://pubmed.ncbi.nlm.nih.gov/advanced/ to create this




In [2]:
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyTool", email=os.getenv("PUBMED_EMAIL"))

# This is the bottom-up query that guided the keywords I wanted.
query = '"geospatial analysis" AND "parkinson* disease"' # n = 3

# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=100))

## Initial Query Results

### Preview query results as a DataFrame

This cell converts `results` (a list of PubMed article objects) into a pandas DataFrame for quick inspection. It shows the first 10 rows and creates `df_results` for further exploration. If `results` is missing/empty, the preview cell prints a message instead.

In [3]:
# Preview `results` as a pandas DataFrame
# This cell converts the `results` list (PubMed article objects) into a DataFrame
# and displays a concise preview (first N rows).

if 'results' not in globals() or not results:
    print("`results` is not defined or empty. Run the query cell above to populate `results`.")
else:
    def _article_to_flat_dict(article):
        # Try to leverage article.toJSON() when available
        try:
            raw = article.toJSON()
            if isinstance(raw, str):
                return json.loads(raw)
            if isinstance(raw, dict):
                return raw
        except Exception:
            pass

        # Fallback: extract common attributes safely
        return {
            "pubmed_id": getattr(article, "pubmed_id", None),
            "title": getattr(article, "title", None),
            "publication_date": str(getattr(article, "publication_date", "") or ""),
            "keywords": getattr(article, "keywords", None),
            "abstract": getattr(article, "abstract", None),
        }

    df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])


In [5]:
# Pick nice default preview columns (if present)
preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"] if c in df_results.columns]

# Make output readable in the notebook
#pd.set_option('display.max_colwidth', 200)
n = 10
print(f"Showing first {min(n, len(df_results))} of {len(df_results)} results (variable: df_results)")
display(df_results[preview_cols].head(n))

Showing first 3 of 3 results (variable: df_results)


,title,abstract,publication_date,keywords
0,Geospatial Analysis of Persons with Movement Disorders Living in Underserved Regions.,Movement disorders persons from underserved areas have increased barriers to access tertiary care. There is currently limited data on the geographic and demographic profile of movement disorders p...,2021-09-14,"[Movement Disorders, geography, spatial analysis, underserved]"
1,Geospatial analysis of individual-based Parkinson's disease data supports a link with air pollution: A case-control study.,"The etiology of Parkinson's disease (PD) remains unknown. To approach the issue of PD's risk factors from a new perspective, we hypothesized that coupling the geographic distribution of PD with sp...",2021-01-22,"[Air pollution, Environment, Epidemiology, Parkinson's disease, Prevalence, Spatial dependence]"
2,Geospatial Analysis of Environmental Atmospheric Risk Factors in Neurodegenerative Diseases: A Systematic Review.,"Despite the vast evidence on the environmental influence in neurodegenerative diseases, those considering a geospatial approach are scarce. We conducted a systematic review to identify studies con...",2020-11-19,"[environment, epidemiology, geospatial, neurodegenerative, systematic review]"


## Refining the Query
I'll use NLP to identify the keywords to expand the initial query on.

In [6]:
# Combine fields into single text per document
# NOTE: make checks robust against numpy arrays / lists / pandas NA (avoid ambiguous truth values)

def _to_text(row, cols=('title', 'abstract', 'keywords')):
    parts = []
    for c in cols:
        # row might be a dict (to_dict orient='records') or a pandas Series
        if isinstance(row, dict):
            val = row.get(c, '')
        else:
            # Series-like
            val = row.get(c, '') if c in row else ''

        # Handle common NA / empty cases safely (avoid pd.isna() directly in an if)
        if val is None:
            val = ''
        elif isinstance(val, float) and np.isnan(val):
            val = ''
        elif isinstance(val, (list, tuple)):
            # join list-like values
            if len(val) == 0:
                val = ''
            else:
                val = ' '.join(map(str, val))
        elif isinstance(val, (np.ndarray,)):
            # convert numpy arrays to list then join
            if val.size == 0:
                val = ''
            else:
                val = ' '.join(map(str, val.tolist()))
        else:
            # keep whatever string representation
            val = '' if (isinstance(val, float) and np.isnan(val)) else str(val)

        parts.append(val)

    # Only return joined string from non-empty parts
    return ' '.join([p for p in parts if p])

# Small helper to normalize candidate terms when comparing
def _normalize(s):
    s = re.sub(r"[^a-z0-9\s]"," ", s.lower())
    s = re.sub(r"\s+"," ", s).strip()
    return s


In [7]:
# Combine relevant text columns into a single corpus
try:
    corpus = [_to_text(r) for r in df_results.to_dict(orient='records')]
    print(f"Built corpus with {len(corpus)} documents")
    if corpus:
        print("Sample (first document, first 300 chars):\n", corpus[0][:300])
except Exception as e:
    # Provide a helpful debugging hint rather than failing silently
    raise RuntimeError("Failed to build corpus from df_results — check the data types in your text columns") from e

Built corpus with 3 documents
Sample (first document, first 300 chars):
 Geospatial Analysis of Persons with Movement Disorders Living in Underserved Regions. Movement disorders persons from underserved areas have increased barriers to access tertiary care. There is currently limited data on the geographic and demographic profile of movement disorders persons from unders


In [8]:
# Parse out current query words/phrases to avoid suggesting exact duplicates
existing_query_terms = set([_normalize(t) for t in re.findall(r"[A-Za-z*]+(?:\s+[A-Za-z*]+)*", query)])

### TF-IDF and N-gram Frequency Calculation
In information retrieval, tf–idf (term frequency–inverse document frequency, TF*IDF, TFIDF, TF–IDF, or Tf–idf) is a measure of importance of a word to a document in a collection or corpus, adjusted for the fact that some words appear more frequently in general.
https://en.wikipedia.org/wiki/Tf%E2%80%93idf

In [9]:
# TF-IDF over 1..3 grams
tfidf = TfidfVectorizer(ngram_range=(1,3), stop_words='english', max_df=0.9)
X = tfidf.fit_transform(corpus)
feature_names = tfidf.get_feature_names_out()

# Sum TF-IDF across rows to get corpus-level importance
scores = np.asarray(X.sum(axis=0)).ravel()

tfidf_df = pd.DataFrame({'term': feature_names, 'tfidf': scores})
#print(f"Computed TF-IDF for {len(tfidf_df)} terms (1-3 grams)")
#print(tfidf_df.head(10))

### n-gram
An *n*-gram is a sequence of *n* adjacent symbols in a particular order. The symbols may be *n* adjacent letters, syllables, or rarely whole words found in a language dataset; or adjacent phonemes extracted from a speech-recording dataset, or adjacent base pairs extracted from a genome. They are collected from a text corpus or speech corpus. https://en.wikipedia.org/wiki/N-gram

In [10]:
# Raw frequency counts for same n-grams
cv = CountVectorizer(ngram_range=(1,3), stop_words='english')
Y = cv.fit_transform(corpus)
freqs = np.asarray(Y.sum(axis=0)).ravel()
cv_terms = cv.get_feature_names_out()

freq_df = pd.DataFrame({'term': cv_terms, 'count': freqs})
#print(f"Computed raw counts for {len(freq_df)} terms (1-3 grams)")
#print(freq_df.head(10))

In [11]:
# Merge the two recommendation signals
cand = tfidf_df.merge(freq_df, on='term', how='outer').fillna(0)

# Add a combined score that balances TF-IDF and frequency
cand['score'] = cand['tfidf'] * 0.7 + (cand['count'] / (cand['count'].max() + 1e-9)) * 0.3

# Add ngram length (for grouping / filtering)
cand['ngram_len'] = cand['term'].str.count(' ') + 1

# Normalize terms for filtering against the query
cand['term_norm'] = cand['term'].apply(_normalize)

# Filter out terms included in the query and short stopwords
cand = cand[~cand['term_norm'].isin(existing_query_terms)]

# Exclude single characters and pure numbers
cand = cand[cand['term'].str.len() > 2]
cand = cand[~cand['term'].str.match(r"^\d+$")]

In [14]:
# Show top candidates for each n-gram size
top_n = 20
results_unified = cand.sort_values('score', ascending=False).head(top_n)

print(f"Found {len(cand)} candidate terms; showing top {len(results_unified)} overall")

# Display grouped results with helpful columns
display_cols = ['term', 'ngram_len', 'count', 'tfidf', 'score']
print('\nTop candidates (combined score):')
display(results_unified[display_cols].reset_index(drop=True))

# Also show top single-word and multi-word candidates separately
for n in (1,2):
    section = cand[cand['ngram_len']==n].sort_values('score', ascending=False).head(10)
    if not section.empty:
        print(f"\nTop {len(section)} ngram_len={n} candidates:")
        display(section[display_cols].reset_index(drop=True))

# Provide a simple function to return a list of recommended terms to extend the query
def suggest_expansions(n=20, min_count=1):
    s = cand[cand['count'] >= min_count].sort_values('score', ascending=False)
    return list(s['term'].head(n))

Found 1039 candidate terms; showing top 20 overall

Top candidates (combined score):


,term,ngram_len,count,tfidf,score
0,underserved,1,9,0.311687,0.463635
1,neurodegenerative,1,6,0.352000,0.410037
2,spatial,1,7,0.189831,0.323791
3,movement,1,6,0.207791,0.309090
4,movement disorders,2,6,0.207791,0.309090
5,persons,1,6,0.207791,0.309090
6,disorders,1,6,0.207791,0.309090
7,environmental,1,5,0.205719,0.280367
8,prevalence,1,5,0.179144,0.261764
9,population,1,5,0.134423,0.230460



Top 10 ngram_len=1 candidates:


,term,ngram_len,count,tfidf,score
0,underserved,1,9,0.311687,0.463635
1,neurodegenerative,1,6,0.352000,0.410037
2,spatial,1,7,0.189831,0.323791
3,movement,1,6,0.207791,0.309090
4,disorders,1,6,0.207791,0.309090
5,persons,1,6,0.207791,0.309090
6,environmental,1,5,0.205719,0.280367
7,prevalence,1,5,0.179144,0.261764
8,population,1,5,0.134423,0.230460
9,review,1,4,0.160191,0.221225



Top 10 ngram_len=2 candidates:


,term,ngram_len,count,tfidf,score
0,movement disorders,2,6,0.207791,0.309090
1,air pollution,2,4,0.143315,0.209412
2,uf nfind,2,4,0.138527,0.206060
3,systematic review,2,3,0.176000,0.205018
4,environmental atmospheric,2,3,0.176000,0.205018
5,neurodegenerative diseases,2,3,0.176000,0.205018
6,risk factors,2,3,0.116484,0.163357
7,spatial dependence,2,3,0.107486,0.157059
8,pd prevalence,2,3,0.107486,0.157059
9,underserved persons,2,3,0.103896,0.154545


## Query with new criteria

In [ ]:
# This is the expanded query using the NLP-driven term rankings. n = 6,335
# Adding exclusion criteria based on scope of research question. 
query = '("parkinson* disease" OR "neurodegenerative disease") AND ("geospatial" OR "spatial dependence" OR "spatiotemporal" OR "geographic" OR "environment*" OR "atmospheric") AND ("pollution" OR "chemical" OR "pesticide") NOT ("pathology" OR "treatment" OR "therapy" OR "intervention" OR "physiology" OR "monitoring" OR "biosensor") NOT ("plant" OR "in vitro" OR "animal" OR "molecular" OR "protein" OR "mice") NOT ("parkinson* disease model" OR "respiratory" OR "resistance training" OR "aggression")' # n = 326, 138 in last 5 years

In [33]:
# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=500))
df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])

# Limit to last 5 years
df_results['publication_date'] = pd.to_datetime(df_results['publication_date'], format='ISO8601')
start_date = '2020-01-01'
df_filtered = df_results[df_results['publication_date'] >= start_date]

# Pick nice default preview columns (if present)
preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"] if c in df_results.columns]

# Make output readable in the notebook
n = 5
print('Total number of articles:', df_results.shape[0]) # get number of rows
print('Total articles in last 5 years:', df_filtered.shape[0]) # get number of rows after date filter
display(df_filtered[preview_cols].head(n))

Total number of articles: 326
Total articles in last 5 years: 156


,title,abstract,publication_date,keywords
0,Air pollution and disease progression in a University of Michigan amyotrophic lateral sclerosis cohort.,"Amyotrophic lateral sclerosis (ALS) is a rare, fatal, neurodegenerative disease without effective treatments. Therefore, identifying modifiable risk factors to slow disease progression is importan...",2025-11-25,"[ALSFRS-R, Air pollution, Amyotrophic lateral sclerosis, Black carbon, Nitrate, Ozone, PM(2.5)]"
1,Inhibition of Mitochondrial Complex III Causes Dopaminergic Neurodegeneration by Redox Stress in,"Environmental factors including chemical exposures are important contributors to Parkinson's disease (PD). Nearly all well-validated chemicals involved in PD affect mitochondria, and the great maj...",2025-11-24,[]
2,Air pollution and Parkinson's disease: A prospective cohort study with sex-stratified analysis in the UK biobank.,"Air pollution has been suggested as a potential environmental risk factor for Parkinson's disease (PD), but findings remain inconsistent, and sex-specific effects are understudied. This study exam...",2025-11-21,"[Air Pollution, Nitrogen Dioxide, Parkinson’s Disease, Particulate Matter, Sex Difference]"
3,Do microplastics play a role in the pathogenesis of neurodegenerative diseases? Shared pathophysiological pathways for Alzheimer's and Parkinson's disease.,"The widespread presence of microplastics (MPs) in the environment has raised significant concerns about their potential impact on human health. As of 2023, the Ocean Conservancy estimates that adu...",2025-11-18,"[Alzheimer’s disease, Microplastics, Neuro-pathophysiology, Neurodegenerative diseases, Parkinson’s disease, Plastic environmental pollution]"
4,Environmental toxins in neurodegeneration - a narrative review.,"As the global incidence of neurodegenerative disorders rises at a rate beyond what can be attributed solely to population aging, the role of modifiable risk factors has come into research spotligh...",2025-11-18,"[Air pollution, Alzheimer’s disease, Amyotrophic lateral sclerosis, Environmental toxins, Heavy metals, Neurodegeneration, Parkinson’s disease, Pesticides, Solvents]"


# Cleaning articles

In [37]:
# Removing duplicates in title
df_filtered = df_filtered.drop_duplicates(subset=['title'])

# Removing if pubmed_id is empty
df_filtered = df_filtered.dropna(subset=['pubmed_id'])

# Remove if doi is missing
df_filtered = df_filtered.dropna(subset=['doi'])

# Preview cleaned df
n = 5
print('Total articles after cleaning:', df_filtered.shape[0]) # get number of rows after cleaning
display(df_filtered[preview_cols].head(n))

Total articles after cleaning: 154


,title,abstract,publication_date,keywords
0,Air pollution and disease progression in a University of Michigan amyotrophic lateral sclerosis cohort.,"Amyotrophic lateral sclerosis (ALS) is a rare, fatal, neurodegenerative disease without effective treatments. Therefore, identifying modifiable risk factors to slow disease progression is importan...",2025-11-25,"[ALSFRS-R, Air pollution, Amyotrophic lateral sclerosis, Black carbon, Nitrate, Ozone, PM(2.5)]"
1,Inhibition of Mitochondrial Complex III Causes Dopaminergic Neurodegeneration by Redox Stress in,"Environmental factors including chemical exposures are important contributors to Parkinson's disease (PD). Nearly all well-validated chemicals involved in PD affect mitochondria, and the great maj...",2025-11-24,[]
2,Air pollution and Parkinson's disease: A prospective cohort study with sex-stratified analysis in the UK biobank.,"Air pollution has been suggested as a potential environmental risk factor for Parkinson's disease (PD), but findings remain inconsistent, and sex-specific effects are understudied. This study exam...",2025-11-21,"[Air Pollution, Nitrogen Dioxide, Parkinson’s Disease, Particulate Matter, Sex Difference]"
3,Do microplastics play a role in the pathogenesis of neurodegenerative diseases? Shared pathophysiological pathways for Alzheimer's and Parkinson's disease.,"The widespread presence of microplastics (MPs) in the environment has raised significant concerns about their potential impact on human health. As of 2023, the Ocean Conservancy estimates that adu...",2025-11-18,"[Alzheimer’s disease, Microplastics, Neuro-pathophysiology, Neurodegenerative diseases, Parkinson’s disease, Plastic environmental pollution]"
4,Environmental toxins in neurodegeneration - a narrative review.,"As the global incidence of neurodegenerative disorders rises at a rate beyond what can be attributed solely to population aging, the role of modifiable risk factors has come into research spotligh...",2025-11-18,"[Air pollution, Alzheimer’s disease, Amyotrophic lateral sclerosis, Environmental toxins, Heavy metals, Neurodegeneration, Parkinson’s disease, Pesticides, Solvents]"


# Export Query Results

In [38]:
# Export cleaned results to CSV
output_csv_path = Path("pubmed_results_cleaned_2025.csv")
df_filtered.to_csv(output_csv_path, index=False)
print(f"Exported cleaned results to {output_csv_path.resolve()}")

Exported cleaned results to /workspaces/sysrev/pubmed_results_cleaned_2025.csv
